In [26]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from scipy.stats import multivariate_normal, norm

snp = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CSPX ETF Stock Price History.csv")
china = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CNYA ETF Stock Price History.csv")
em = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "EIMI ETF Stock Price History.csv")
gold = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XAD5 ETF Stock Price History.csv")
india = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "INR ETF Stock Price History.csv")
mscieurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XMEU ETF Stock Price History.csv")
smallcapeurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "SXRJ ETF Stock Price History.csv")

dfs = [snp, china, em, gold, india, mscieurope, smallcapeurope]
date_sets = [set(df["Date"]) for df in dfs]

print('lens:', [len(df) for df in dfs])

# Find the intersection of all date sets
common_dates = set.intersection(*date_sets)

# Filter each DataFrame to include only common dates
dfs = [df[df["Date"].isin(common_dates)] for df in dfs]

N = len(dfs)
T = len(dfs[0]) - 1

returns_all = np.empty((T, N))
sorted_returns_all = np.empty((T, N))
volumes_all = np.empty((T, N))
for i, df in enumerate(dfs):
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce")  # Handles str/NaN/object
    returns = df["Price"].pct_change().values[1:]
    returns_all[:, i] = returns
    sorted_returns_all[:, i] = sorted(returns)
    volumes_all[:, i] = df["Vol_clean"].values[1:]

all_dates = dfs[0]["Date"].values[1:]
volumes_all = np.nan_to_num(volumes_all, nan=0.0)

lens: [3925, 2693, 2990, 3943, 4941, 4527, 4052]


C:\Users\tobia\AppData\Local\Temp\ipykernel_16168\1442181442.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Price"] = pd.to_numeric(df["Price"], errors="coerce")  # Handles str/NaN/object


In [16]:
def perform_validation(w, validation_data):
    # validation data is shape (T, N)
    
    # perform metrics
    total_return = validation_data @ w # shape (T,)
    cumulative_returns = np.cumprod(total_return + 1) - 1
    risk_free = 0.0
    std = np.std(total_return)
    downside_std = np.std(total_return[total_return > 0])
    sharpe = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * std)
    sortino = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * downside_std)    

    alpha = 0.10

    var_alpha = np.quantile(total_return, alpha)
    cvar = total_return[total_return <= var_alpha].mean()
    mean_return = np.mean(total_return)
    return var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return
    

In [17]:
def lower_part_mean(matrix):
    n, _ = matrix.shape
    s = 0.0
    for i in range(1,n):
        for j in range(i):
            s += matrix[i,j]
    N = (n-1)*n/2
    return s / N

def compute_statistics_rolling(returns, window_size, stepsize):
    print('returns:'); print(returns)
    all_corrs = []
    all_volas = []
    T, N = returns.shape
    w = np.ones(N) / N
    print('T:', T)
    print('testetst')
    for idx in range(0, T - window_size, stepsize):
        section = returns[idx:idx+window_size,:]
        print('section shape:', section.shape)
        corr = np.corrcoef(section, rowvar=False) # (N,N)
        print('appending', lower_part_mean(corr))
        all_corrs.append(lower_part_mean(corr))
        returns_total = section @ w
        print('returns_total:', returns_total)
        std = np.std(returns_total)
        all_volas.append(std)
    
    all_corrs = np.array(all_corrs)
    all_volas = np.array(all_volas)
    print('result:'); print(all_corrs)
    return all_corrs, all_volas


In [18]:
# run

corrs, volas = compute_statistics_rolling(returns_all, 90, 30)

print('mean corrs:', np.mean(corrs))
print('mean volas:', np.mean(volas))
print('highest corrs:', np.max(corrs))
print('lowest corrs:', np.min(corrs))
print('lowest volas:', np.min(volas))
print('highest volas:', np.max(volas))
# results:
# based on this, determine threshold for correlation and volatility

vola_thresh = np.percentile(volas, 75)
corr_thresh = np.percentile(corrs, 75)

returns:
[[ 0.00531537  0.01223776  0.01829268 ...  0.01805206  0.01167487
   0.0165625 ]
 [ 0.01289301  0.01208981  0.01753636 ...  0.01690722  0.02811201
   0.02782047]
 [ 0.0076485   0.01023891  0.00861707 ...  0.01175994  0.00702202
  -0.0013459 ]
 ...
 [ 0.01019448  0.03183521  0.01616719 ...  0.01857585  0.01546781
   0.01932859]
 [-0.00041401 -0.03811252 -0.01164144 ...  0.02553191  0.00520124
   0.01041667]
 [-0.0057986   0.00188679  0.00078524 ... -0.0017783  -0.00061599
  -0.00413606]]
T: 2526
testetst
section shape: (90, 7)
appending 0.5125406050943999
returns_total: [ 1.21390880e-02  2.56297651e-02  9.80273266e-03 -4.64520375e-03
 -1.73647614e-03  6.21773253e-03  1.01113437e-02  7.86574005e-03
 -1.86355336e-02  6.39879203e-03  2.83296486e-03  1.01599801e-02
 -1.37725969e-02  3.28829889e-02  9.50225006e-03  2.92485384e-04
  2.03606765e-03 -6.24396729e-03 -1.57912574e-03  6.39079369e-04
 -9.17700612e-03  6.03244470e-03 -1.16839290e-02  1.12677481e-03
 -9.87316715e-04 -5.94482

In [30]:
import tensorflow as tf

# train weights
alpha = 0.05   # tail probability for CVaR
lambda_ = 0.5  # CVaR penalty
learning_rate = 0.01
gamma = 0.02
num_steps = 2000
beta = 0.1 # penalty volatility

def gradient(train_data, corr_thresh, vol_thresh):

    # returns (weights, mode)
    # where mode = 0 if returned normal (optimal) weights,
    # 1 = if returned equal because of correlations
    # 2 = if returned equal because of volatility

    # train data: (T,N)
    w = tf.Variable(tf.ones([N], dtype=tf.float32) / N)
    t = tf.Variable(0.0, dtype=tf.float32)  # VaR threshold

    optimizer = tf.keras.optimizers.Adam(learning_rate)

    X_tf = tf.convert_to_tensor(train_data, dtype=tf.float32) # (T,N)
    #cov_tf = tf.convert_to_tensor(cov, dtype=tf.float32)  # shape (N, N)

    corr = np.corrcoef(train_data, rowvar=False)
    mean_corr = lower_part_mean(corr)
    print('mean corr:', mean_corr)

    if mean_corr > corr_thresh:
        return np.ones(N) / N, 1

    for step in range(num_steps):
        with tf.GradientTape() as tape:
            # enforce sum-to-1 via softmax
            w_normalized = tf.nn.softmax(w)
            w_col = tf.reshape(w_normalized, [N, 1])  # shape (N,1)

            # portfolio returns
            portfolio_returns = tf.matmul(X_tf, tf.reshape(w_col, [-1,1])) # (T)
            losses = -portfolio_returns
            cvar = t + (1 / (1 - alpha)) * tf.reduce_mean(tf.nn.relu(losses - t))

            import tensorflow as tf

# train weights
alpha = 0.05   # tail probability for CVaR
lambda_ = 0.5  # CVaR penalty
learning_rate = 0.01
gamma = 0.02
num_steps = 2000
beta = 0.1 # penalty volatility

def gradient(train_data, vol_data, corr_thresh, vol_thresh):

    # returns (weights, mode)
    # where mode = 0 if returned normal (optimal) weights,
    # 1 = if returned equal because of correlations
    # 2 = if returned equal because of volatility

    # train data: (T,N)
    w = tf.Variable(tf.ones([N], dtype=tf.float32) / N)
    t = tf.Variable(0.0, dtype=tf.float32)  # VaR threshold

    optimizer = tf.keras.optimizers.Adam(learning_rate)

    X_tf = tf.convert_to_tensor(train_data, dtype=tf.float32) # (T,N)
    V_tf = tf.convert_to_tensor(vol_data, dtype=tf.float32) # (T,N)
    #cov_tf = tf.convert_to_tensor(cov, dtype=tf.float32)  # shape (N, N)

    corr = np.corrcoef(train_data, rowvar=False)
    mean_corr = lower_part_mean(corr)
    print('mean corr:', mean_corr)

    if mean_corr > corr_thresh:
        return np.ones(N) / N, 1

    for step in range(num_steps):
        with tf.GradientTape() as tape:
            # enforce sum-to-1 via softmax
            w_normalized = tf.nn.softmax(w)
            w_col = tf.reshape(w_normalized, [N, 1])  # shape (N,1)

            # portfolio returns
            portfolio_returns = tf.matmul(X_tf, tf.reshape(w_col, [-1,1])) # (T)
            losses = -portfolio_returns
            cvar = t + (1 / (1 - alpha)) * tf.reduce_mean(tf.nn.relu(losses - t))

            weighted_volume = tf.matmul(V_tf, tf.reshape(w_col, [-1,1])) # (T)

            momentum = tf.reduce_mean(X_tf[-60:], axis=0)
            
            vol_short = tf.reduce_mean(weighted_volume[-10:])
            vol_long  = tf.reduce_mean(weighted_volume[-60:])
            volume_signal = vol_short / (vol_long + 1e-8)
            #print('volume signal:', volume_signal)
            mu = tf.convert_to_tensor(momentum * volume_signal)
            #print('mu:', mu)

            mu = (mu - np.mean(mu)) / (np.std(mu) + 1e-8) # shape (N,)
            #print('mu second:', mu)
            penalty = gamma * tf.reduce_sum(tf.math.square(w_col))

            objective = -(tf.tensordot(w_normalized, mu, axes=1) - lambda_ * cvar) + penalty

        # compute gradients
        grads = tape.gradient(objective, [w, t])
        optimizer.apply_gradients(zip(grads, [w, t]))

    # after the optimization loop
    optimal_w = tf.nn.softmax(w).numpy()

    # compute volatility
    combined_returns = train_data @ optimal_w # (T,)
    volatility = np.std(combined_returns)

    print('volatility:', volatility)

    if volatility > vol_thresh:
        return np.ones(N) / N, 2

    return optimal_w, 0

In [20]:
def rolling_window(window_size, stepsize, validation_size, returns_data, volume_data, corr_thresh, vol_thresh):
    # data is (T,N)

    sharpes_w_opt= []
    sharpes_w_eqw = []

    mean_return_w_opt = []
    mean_return_w_eqw = []

    sortinos_w_opt = []
    sortinos_w_eqw = []

    vars_w_opt = []
    vars_w_eqw = []

    cvars_w_opt = []
    cvars_w_eqw = []

    T, N = returns_data.shape
    weights_opt = []

    mode_counter = [0, 0, 0] # normal, corr, vol

    for idx in range(0, T - window_size - validation_size, stepsize):
        train_data_returns = returns_data[idx:idx+window_size]
        val_data_returns = returns_data[idx+window_size:idx+window_size+validation_size]
        train_data_volume = volume_data[idx:idx+window_size]
        #val_data_volume = volume_data[idx+window_size:idx+window_size+validation_size]
        cov = np.cov(train_data_returns, rowvar=False)

        w, mode = gradient(train_data_returns, train_data_volume, corr_thresh, vol_thresh)
        weights_opt.append(w)
        mode_counter[mode] += 1

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return
        ) = perform_validation(w, val_data_returns)

        sharpes_w_opt.append(sharpe)
        sortinos_w_opt.append(sortino)
        vars_w_opt.append(var_alpha)
        cvars_w_opt.append(cvar)
        mean_return_w_opt.append(mean_return)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return
        ) = perform_validation(np.ones(N) / N, val_data_returns)

        sharpes_w_eqw.append(sharpe)
        sortinos_w_eqw.append(sortino)
        vars_w_eqw.append(var_alpha)
        cvars_w_eqw.append(cvar)
        mean_return_w_eqw.append(mean_return)

    df = pd.DataFrame({
        "Sharpes": [np.mean(sharpes_w_eqw), np.mean(sharpes_w_opt)],
        "Sortinos": [np.mean(sortinos_w_eqw), np.mean(sortinos_w_opt)],
        "Vars": [np.mean(vars_w_eqw), np.mean(vars_w_opt)],
        "Cvars": [np.mean(cvars_w_eqw), np.mean(cvars_w_opt)],
        "MeanReturn": [np.mean(mean_return_w_eqw), np.mean(mean_return_w_opt)],
    }, index=["Equal weights", "Return + CVaR"])

    weights_opt = np.array(weights_opt)

    return df, np.mean(weights_opt, axis=0), mode_counter

In [ ]:
df, weights_opt, mode_counter = rolling_window(
    window_size=252*2,
    stepsize=50,
    validation_size=252,
    returns_data=returns_all,
    volume_data=volumes_all,
    corr_thresh=corr_thresh,
    vol_thresh=vola_thresh
)
print('chosen weights')
print(weights_opt)
print()
df

print('mean weights optimized:', weights_opt)
s = sum(mode_counter)
print('pct of weights chosen optimally:', mode_counter[0]/s)
print('pct of weights chosen bec. of correlation:', mode_counter[1]/s)
print('pct of weights chosen bec. of volatility:', mode_counter[2]/s)


mean corr: 0.3705022877332705
volume signal: tf.Tensor(0.41371983, shape=(), dtype=float32)
mu: tf.Tensor(
[-0.0003553   0.0001363  -0.00025408 -0.00051307 -0.00060993 -0.00016762
 -0.00024187], shape=(7,), dtype=float32)
mu second: tf.Tensor(
[-0.30478188  1.873182    0.14366806 -1.0037192  -1.4328444   0.52670914
  0.19778605], shape=(7,), dtype=float32)


TypeError: cannot unpack non-iterable NoneType object

In [ ]:



def stress_test(df_data, start_train, end_train, end_valid):

    val_returns_total = []
    train_returns_total = []

    modes = np.zeros(3)

    for df in df_data:
        df_datetime = df.copy()
        df_datetime["Date"] = pd.to_datetime(df_datetime["Date"])
        train = df_datetime[
            (df_datetime["Date"] >= pd.to_datetime(start_train))
            & (df_datetime["Date"] <= pd.to_datetime(end_train))]
        val = df_datetime[
            (df_datetime["Date"] >= pd.to_datetime(end_train))
            & (df_datetime["Date"] <= pd.to_datetime(end_valid))
        ]

        train_returns = train["Close"].pct_change().values[1:]
        val_returns = val["Close"].pct_change().values[1:]

        val_returns_total.append(val_returns)
        train_returns_total.append(train_returns)
        
    train_returns_total = np.array(train_returns_total).T # shape (num_days, N = num_assets)
    val_returns_total = np.array(val_returns_total).T

    w_opt, mode_idx = gradient(train_returns_total, corr_thresh, vola_thresh)
    print('chosen weights optimal:', w_opt)
    modes[mode_idx] += 1
    
    return perform_validation(w_opt, val_returns_total), perform_validation(np.ones(N) / N, val_returns_total), modes


In [ ]:
start_train = "2024-01-01"
end_train = "2025-01-01"
end_valid = "2025-04-01"

results_optimized, results_eqw, modes = stress_test(
    dfs, start_train, end_train, end_valid
)
var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return = results_optimized

print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print()

var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return = results_eqw

print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print()

s = sum(mode_counter)
print('pct of weights chosen optimally:', mode_counter[0]/s)
print('pct of weights chosen bec. of correlation:', mode_counter[1]/s)
print('pct of weights chosen bec. of volatility:', mode_counter[2]/s)


fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns))), y=cumulative_returns))
fig.show()